# 规则引擎模块演示 (Rules Module)

本notebook演示hscredit库中规则引擎模块的功能，包括规则定义、评估和组合。

In [ ]:
# 添加项目路径
import sys
import os
sys.path.append('../')

# 初始化设置
from hscredit.utils import init_setting
init_setting(seed=42)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import os

# 加载数据
data_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/hscredit.xlsx'
df = pd.read_excel(data_path)
print(f"数据形状: {df.shape}")
print(f"\n列名: {df.columns.tolist()}")

## 1. 导入规则引擎模块

In [ ]:
from hscredit.core.rules import Rule, get_columns_from_query, optimize_expr, beautify_expr

print("规则引擎模块导入成功！")

## 2. 创建简单规则

使用Rule类创建风控规则。

In [ ]:
# 选择数值特征进行规则创建
demo_feature = df.select_dtypes(include=[np.number]).columns[0]
print(f"演示特征: {demo_feature}")

# 创建规则: 特征值大于某阈值
mean_val = df[demo_feature].mean()
rule = Rule(f"{demo_feature} > {mean_val}", name=f"高于均值规则")

print(f"规则: {rule.expr}")
print(f"规则名称: {rule.name}")

In [ ]:
# 规则预测
result = rule.predict(df)
print(f"规则命中样本数: {result.sum()}")
print(f"规则命中率: {result.mean()*100:.2f}%")

## 3. 组合规则

使用 & (与)、| (或)、~ (非) 运算符组合规则。

In [ ]:
# 选择多个特征
features = df.select_dtypes(include=[np.number]).columns[:3].tolist()
print(f"选择特征: {features}")

# 创建多个规则
rule1 = Rule(f"{features[0]} > {df[features[0]].quantile(0.75)}", name="高值规则")
rule2 = Rule(f"{features[1]} < {df[features[1]].quantile(0.25)}", name="低值规则")

# AND组合
rule_and = rule1 & rule2
print(f"AND规则: {rule_and.expr}")

# OR组合
rule_or = rule1 | rule2
print(f"OR规则: {rule_or.expr}")

# NOT组合
rule_not = ~rule1
print(f"NOT规则: {rule_not.expr}")

In [ ]:
# 评估组合规则
print("规则评估结果:")
for name, r in [('AND', rule_and), ('OR', rule_or), ('NOT', rule_not)]:
    result = r.predict(df)
    print(f"  {name}: 命中{result.sum()}样本, 命中率{result.mean()*100:.2f}%")

## 4. 复杂规则表达式

创建包含多个条件的复杂规则。

In [ ]:
# 创建复杂规则
rule_complex = Rule(
    f"({features[0]} > {df[features[0]].mean()}) & ({features[1]} < {df[features[1]].median()})",
    name="复杂规则"
)

print(f"复杂规则: {rule_complex.expr}")

result_complex = rule_complex.predict(df)
print(f"命中样本数: {result_complex.sum()}")
print(f"命中率: {result_complex.mean()*100:.2f}%")

## 5. 规则与目标变量交叉分析

分析规则命中样本的目标分布。

In [ ]:
# 规则命中与目标变量交叉分析
rule_result = rule_complex.predict(df)
cross_tab = pd.crosstab(rule_result, df['FPD15'], margins=True)
print("规则命中与FPD15交叉表:")
print(cross_tab)

# 计算规则对坏样本的捕获率
bad_captured = (rule_result & (df['FPD15'] == 1)).sum()
total_bad = (df['FPD15'] == 1).sum()
capture_rate = bad_captured / total_bad * 100
print(f"\n坏样本捕获率: {capture_rate:.2f}%")

## 6. 规则优化和美化

优化和美化规则表达式。

In [ ]:
# 规则表达式优化
expr = "(age > 18) & (income >= 5000) | (credit_score >= 700)"
optimized = optimize_expr(expr)
print(f"原始表达式: {expr}")
print(f"优化后: {optimized}")

# 美化表达式
beautified = beautify_expr(expr)
print(f"美化后: {beautified}")

## 7. 批量规则评估

对多条规则进行批量评估。

In [ ]:
# 创建多条规则
rules_list = []
for i, col in enumerate(features):
    q75 = df[col].quantile(0.75)
    q25 = df[col].quantile(0.25)
    r_high = Rule(f"{col} > {q75}", name=f"高值_{col}")
    r_low = Rule(f"{col} < {q25}", name=f"低值_{col}")
    rules_list.extend([r_high, r_low])

print(f"创建了 {len(rules_list)} 条规则")

# 批量评估
rule_names = []
hit_counts = []
hit_rates = []

for r in rules_list:
    result = r.predict(df)
    rule_names.append(r.name)
    hit_counts.append(result.sum())
    hit_rates.append(result.mean() * 100)

rules_eval_df = pd.DataFrame({
    '规则名称': rule_names,
    '命中数': hit_counts,
    '命中率(%)': hit_rates
})
print("\n规则评估结果:")
print(rules_eval_df.to_string(index=False))

## 8. 保存规则结果

将规则评估结果保存到Excel。

In [ ]:
# 保存规则结果
output_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/rule_results.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    rules_eval_df.to_excel(writer, sheet_name='规则评估', index=False)
    cross_tab.to_excel(writer, sheet_name='交叉分析')

print(f"规则结果已保存至: {output_path}")